# HNSWlib


### 1. hnswlib (Hierarchical NSW) —— 核心代码 `hnswlib_clerc_runner.cpp`

根据 HNSW 论文第四、五节及运行参数，影响曲线的最核心参数如下：

| 参数类别 | 参数名称 | 对应代码/文档变量 | 对 QPS-Accuracy / MRR / Recall 曲线的影响机理 |
| :--- | :--- | :--- | :--- |
| **运行时 (直接旋钮)** | **`ef_search`** | `argv[9]` (文档设为 100) | **决定性超参数**。增大 `ef_search` 会扩展搜索时的动态候选列表（论文 Algorithm 2 中的 `W`），显著提高召回率和 MRR（因为能找到更优的最近邻），但会导致距离计算次数急剧增加，**QPS 直线下降**。这是绘制权衡曲线最核心的调节变量。 |
| **构建时 (质量天花板)** | **`M`** | `argv[7]` (文档设为 16) | 每层每个节点的最大出度。增大 `M` 使图连接更密集，提高索引的“导航”精度（相同 `ef_search` 下 Recall/MRR 更高），但会增大内存访问开销和贪心路径的度数，**降低 QPS**。论文建议 5~48 之间调节。 |
| **构建时 (质量天花板)** | **`ef_construction`** | `argv[8]` (文档设为 200) | 构建索引时的动态候选列表大小。该值越大，构建的图质量越高（更接近 Delaunay 图），使得在**相同的 `ef_search`** 下获得更高的 Recall/MRR。但它不影响搜索时的 QPS，只影响曲线的饱和上限。 |
| **隐式结构参数** | **`M_max0`** (零层连接数) | 代码中默认使用 `M * 2` | 论文第 4.1 节指出，若设置过小（等于 M），高召回率下性能会严重退化；设为 `2*M` 能保证高 Recall 下的搜索效率。若手动调整该值（需改源码），会显著影响高 Recall 区域的 QPS 曲线形态。 |

> **绘图策略**：固定 `M=16`, `ef_construction=200`，**仅调整 `ef_search`**（如从 10 扫到 500），即可完美绘制出 hnswlib 的三条曲线。

---

### 2. HCNNG —— 文档 `README_clerc_large_single.md`

| 参数类别 | 参数名称 | 对应文档命令 | 对曲线的影响机理 |
| :--- | :--- | :--- | :--- |
| **运行时 (直接旋钮)** | **`max_calc`** | `search` 命令第 6 个参数（文档设为 2000） | **唯一直接影响权衡的运行时参数**。它限制了每查询的向量距离计算次数（搜索预算）。预算越大，贪心路由探索范围越广，Recall/MRR 越高，但 QPS 严格线性下降。 |
| **构建时** | **`target_cluster_size`** | `hcnng` 命令第 2 个参数（文档设为 1000） | 影响树的层级划分。较小的目标簇大小会使树更深，虽然能略微改善搜索初期定位，但在高维数据（1024维）下影响远小于 `max_calc`。 |

> **绘图策略**：固定索引参数，**仅调整 `max_calc`**（如从 100 到 5000）绘制曲线。

---

### 额外影响曲线形态的全局因素（代码中可见）

- **搜索线程数 (`search_threads` / `--nworker` / OpenMP 并行)**：在 `hnswlib_clerc_runner` 和 ELPIS 中，增加线程数会显著**提升 QPS**（吞吐量），但在单查询延迟（latency）上可能略有开销。它不会改变算法理论上的 Accuracy/Recall，但会使 **QPS-Accuracy 曲线整体向上平移**。文档中 hnswlib 刻意设置了 `search_threads=1` 来测纯算法延迟。

### 总结：绘制三条曲线的最简参数集

| 算法 | **需调节的运行时参数 (X轴变量)** | **需固定的结构参数** |
| :--- | :--- | :--- |
| **HNSW (hnswlib)** | **`ef_search`** | `M` (建议16), `ef_construction` (建议200) |
| **HCNNG** | **`max_calc`** (搜索预算) | `target_cluster_size` (建议1000) |
| **ELPIS** | **`--nprobes`** (探测叶数) | `--kb` (16), `--Lb` (200), `--L` (100), `--leaf-size` (100) |


In [ ]:
# Compile the C++ runner

g++ /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/hnswlib_clerc_runner.cpp \
  -O3 -std=c++14 -fopenmp \
  -o /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/hnswlib_clerc_runner

In [ ]:
import json
import os
import re
import pandas as pd

def extract_hcnng_metrics(
    base_dir: str 
):
    """
    遍历 HCNNG 结果目录（maxcalc_* 子目录），提取每个 max_calc 下的 BEIR 指标和 QPS，
    汇总为一个 CSV 文件，保存到指定路径。

    参数:
        base_dir (str): 包含 maxcalc_* 子目录的根目录。
    """

    rows = []

    # 遍历所有 maxcalc_* 子目录
    for item in os.listdir(base_dir):
        if not item.startswith("maxcalc_"):
            continue
        sub_dir = os.path.join(base_dir, item)
        if not os.path.isdir(sub_dir):
            continue

        # 提取 max_calc 数值
        match = re.search(r"maxcalc_(\d+)", item)
        if not match:
            continue
        max_calc = int(match.group(1))

        # ---- 读取 JSON 指标 ----
        json_path = os.path.join(sub_dir, "beir_metrics_k100_scored.json")
        if not os.path.isfile(json_path):
            print(f"警告: {json_path} 不存在，跳过")
            continue

        with open(json_path, "r") as f:
            data = json.load(f)

        # 提取 K 值（取第一个）
        k_values = data.get("k_values", [])
        if not k_values:
            print(f"警告: {json_path} 中无 k_values，跳过")
            continue
        K = k_values[0]

        # 手动提取常见指标
        row = {
            "K": K,
            "max_calc": max_calc,
            "Recall@100": data.get("recall", {}).get("Recall@100"),
            "NDCG@100": data.get("ndcg", {}).get("NDCG@100"),
            "MRR@100": data.get("mrr", {}).get("MRR@100"),
            "MAP@100": data.get("map", {}).get("MAP@100"),
            "P@100": data.get("precision", {}).get("P@100"),
            "Hole@100": data.get("hole", {}).get("Hole@100"),
            "Accuracy@100": data.get("accuracy", {}).get("Accuracy@100"),
            "R_cap@100": data.get("recall_cap", {}).get("R_cap@100"),
            "queries_qrels": data.get("queries_qrels"),
            "queries_results": data.get("queries_results"),
        }

        # ---- 读取 QPS ----
        qps_path = os.path.join(sub_dir, "qps_k100_scored.tsv")
        if not os.path.isfile(qps_path):
            print(f"警告: {qps_path} 不存在，跳过")
            continue

        try:
            qps_df = pd.read_csv(qps_path, sep='\t')
            if 'QPS' not in qps_df.columns:
                print(f"警告: {qps_path} 中缺少 QPS 列，跳过")
                continue
            qps_value = qps_df['QPS'].iloc[0]          # 取第一行 QPS
            if pd.isna(qps_value):
                print(f"警告: {qps_path} 中 QPS 值为空，跳过")
                continue
            row["QPS"] = float(qps_value)
        except Exception as e:
            print(f"警告: 读取 {qps_path} 失败 ({e})，跳过")
            continue

        rows.append(row)


    if not rows:
        print("未找到任何有效数据，不生成 CSV")
        return

    # 转换为 DataFrame 并按 max_calc 排序
    df = pd.DataFrame(rows).sort_values("max_calc").reset_index(drop=True)

    # 保存 CSV
    output_csv=os.path.join(base_dir,"summary.csv")
    df.to_csv(output_csv, index=False)
    print(f"汇总结果已保存至: {output_csv}")

In [ ]:
# plot

%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

def plot_qps_metrics(csv_file, topk=None, max_calc_list=None):
    """
    绘制 QPS vs Recall@100 和 QPS vs MRR@100 曲线。
    
    参数:
        csv_file (str): CSV 文件路径。
        topk (int, optional): 只绘制指定 K 的结果（如 100）。
        ndocs_list (list, optional): 只绘制 max_calc 在列表中的行。
    """
    # 读取数据
    df = pd.read_csv(csv_file)

    # 过滤 topk (对应 CSV 中的 K 列)
    if topk is not None:
        df = df[df['K'] == topk]
    
    # 过滤 ndocs (对应 CSV 中的 max_calc 列)
    if max_calc_list is not None:
        df = df[df['max_calc'].isin(max_calc_list)]
    
    # 如果过滤后为空，给出提示
    if df.empty:
        print("筛选后没有数据，请检查参数。")
        return
    
    # 按 QPS 升序排序，使曲线连贯
    df_sorted = df.sort_values("QPS")
    
    # 设置绘图风格
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.size'] = 12
    
    # 创建两个子图，方便比较
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # ---------- 图1: QPS vs Recall@100 ----------
    ax1.plot(df_sorted["QPS"], df_sorted["Recall@100"], 
             marker='o', linestyle='-', linewidth=2, markersize=8, color='b')
    ax1.set_xlabel("QPS (queries/sec)", fontsize=14)
    ax1.set_ylabel("Recall@100", fontsize=14)
    ax1.set_title("QPS vs Recall@100", fontsize=16)
    ax1.grid(True, linestyle='--', alpha=0.7)
    
    # 标注每个点：优先使用 'max_calc' 列，若无则用行号
    if 'max_calc' in df_sorted.columns:
        label_col = 'max_calc'
        label_prefix = 'max_calc='
    else:
        label_col = None
        label_prefix = ''
    
    for idx, row in df_sorted.iterrows():
        if label_col:
            label = f"{label_prefix}{row[label_col]}"
        else:
            label = f"idx{idx}"
        ax1.annotate(label, 
                     xy=(row["QPS"], row["Recall@100"]),
                     xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    # ---------- 图2: QPS vs MRR@100 ----------
    ax2.plot(df_sorted["QPS"], df_sorted["MRR@100"], 
             marker='s', linestyle='-', linewidth=2, markersize=8, color='r')
    ax2.set_xlabel("QPS (queries/sec)", fontsize=14)
    ax2.set_ylabel("MRR@100", fontsize=14)
    ax2.set_title("QPS vs MRR@100", fontsize=16)
    ax2.grid(True, linestyle='--', alpha=0.7)
    
    for idx, row in df_sorted.iterrows():
        if label_col:
            label = f"{label_prefix}{row[label_col]}"
        else:
            label = f"idx{idx}"
        ax2.annotate(label, 
                     xy=(row["QPS"], row["MRR@100"]),
                     xytext=(5, 5), textcoords='offset points', fontsize=9)

    ax1.set_xscale('log')
    ax2.set_xscale('log')
    ax1.set_xticks([1, 10, 100, 1000])
    ax1.set_xticklabels(['$10^0$', '$10^1$', '$10^2$', '$10^3$'])
    ax2.set_xticks([1, 10, 100, 1000])
    ax2.set_xticklabels(['$10^0$', '$10^1$', '$10^2$', '$10^3$'])
    
    plt.tight_layout()
    
    # 保存到 CSV 同目录，文件名与 CSV 相同（扩展名 .png）
    output_dir = os.path.dirname(csv_file)
    base_name = os.path.splitext(os.path.basename(csv_file))[0]
    save_path = os.path.join(output_dir, base_name + ".png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"图片已保存为 '{save_path}'")


# [100, 500, 1000, 2000, 3000, 4000, 5000]

## Clerc-large

In [ ]:
# Run hnswlib on the 1000-query subset

/home/ali/SVR-baselines/runs/hnswlib_clerc_runner \
  /data/ali/clerc-large-single/clerc-large-single_base.fvecs \
  /data/ali/baseline-data/clerc-large-single/inputs/query_1000.fvecs \
  /data/ali/baseline-data/clerc-large-single/hnswlib/index.bin \
  /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/clerc-large-single/hnswlib/ans_k100_scored.tsv \
  1000 \
  100 \
  16 \
  200 \
  100 \
  96 \
  1 \
  > /data1/chenyifeng/MultiVector-Backup/svr-baselines/runs/clerc-large-single/logs/hnswlib_k100_scored.log 2>&1


Argument meaning:

- `1000`: number of queries to run
- `100`: return top-100 neighbors
- `16`: `M`
- `200`: `ef_construction`
- `100`: `ef_search`
- `96`: build threads
- `1`: search threads

Important:

- this runner is C++ only
- index construction may use multiple threads
- search is intentionally one query at a time with one search thread
- the logged `[Search Time]` excludes result-file writing

In [ ]:
# hnswlib Evaluate Recall@10 and Recall@100

python3 /home/ali/SVR-baselines/runs/eval_recall.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/hnswlib/ans_k100_scored.tsv \
  --k 1

python3 /home/ali/SVR-baselines/runs/eval_recall.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/hnswlib/ans_k100_scored.tsv \
  --k 10

python3 /home/ali/SVR-baselines/runs/eval_recall.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/hnswlib/ans_k100_scored.tsv \
  --k 100

In [ ]:
# BEIR-style evaluation

python3 /home/ali/SVR-baselines/runs/eval_beir_metrics.py \
  --groundtruth /data/ali/baseline-data/clerc-large-single/inputs/groundtruth_1000.ivecs \
  --results /home/ali/SVR-baselines/runs/clerc-large-single/hnswlib/ans_k100_scored.tsv \
  --k-values 1 3 5 10 100 \
  --output-json /home/ali/SVR-baselines/runs/clerc-large-single/hnswlib/beir_metrics_k100_scored.json

In [ ]:
# hnswlib QPS

the logged QPS is already the search-only QPS to use.

## Scidocs